<a href="https://colab.research.google.com/github/ascii5833/Deep-Learning-Mastery/blob/main/Codes/Transformers/SelfAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
import torch.nn.functional as F

<h1>Preparing our data </h1>

In [45]:
sentence = "I think Aahan ate the cheese pizza"
#creating  dictionary
dc = {k : v for v, k in enumerate(sorted(sentence.split()))}
print(f'Dictionary is :\n {dc}')

Dictionary is :
 {'Aahan': 0, 'I': 1, 'ate': 2, 'cheese': 3, 'pizza': 4, 'the': 5, 'think': 6}


In [46]:
#map sentence to the required integer indices
idx = [dc[k] for v, k in enumerate(sorted(sentence.split()))]
sentence_t = torch.tensor(idx)
print(sentence_t)

tensor([0, 1, 2, 3, 4, 5, 6])


<h1> Creating an embedding </h1>

In [47]:
# we assume a large vocabulary size
vocab_size = 40000
#reproducability
torch.manual_seed(123)
#since we have no training we shall detach the embeddings
emb = nn.Embedding(vocab_size, 4)
sentence_embeddings = emb(sentence_t).detach()
print(f'Embedded sentences are: \n {sentence_embeddings}')



Embedded sentences are: 
 tensor([[ 0.3374, -0.1778, -0.3035, -0.5880],
        [ 0.3486,  0.6603, -0.2196, -0.3792],
        [ 0.7671, -1.1925,  0.6984, -1.4097],
        [ 0.1794,  1.8951,  0.4954,  0.2692],
        [-0.0770, -1.0205, -0.1690,  0.9178],
        [ 1.5810,  1.3010,  1.2753, -0.2010],
        [ 0.4965, -1.5723,  0.9666, -1.1481]])


<h1> Self-Attention Mechanism </h1>

In [48]:
D = sentence_embeddings.shape[1]
print(f'Embedding size is {D}')
#dimensions for the query, key and value vectors
D_q, D_k, D_v = 2, 2, 4

Embedding size is 4


In [49]:
#weights for query, keys and values
W_query = nn.Parameter(torch.rand(D, D_q))
W_keys = nn.Parameter(torch.rand(D, D_k))
W_values = nn.Parameter(torch.rand(D, D_v))

queries = sentence_embeddings @ W_query
keys = sentence_embeddings @ W_keys
values = sentence_embeddings @ W_values


<h1>Attention weights </h1>

In [50]:
Q_dot_K = queries @ keys.T
norm_Q_dot_K = Q_dot_K / math.sqrt(D_k)
#-1 for softmax along final dimension, which is the column
attention = F.softmax(norm_Q_dot_K, dim = -1)
#the attention weights
print(attention)



tensor([[0.1846, 0.1593, 0.1518, 0.1101, 0.1776, 0.0617, 0.1549],
        [0.1530, 0.1493, 0.1398, 0.1420, 0.1596, 0.1130, 0.1433],
        [0.2300, 0.1627, 0.1634, 0.0669, 0.1904, 0.0226, 0.1639],
        [0.0729, 0.0942, 0.0909, 0.1831, 0.0862, 0.3808, 0.0918],
        [0.0925, 0.1081, 0.1178, 0.1585, 0.0936, 0.3156, 0.1138],
        [0.0639, 0.0850, 0.0846, 0.1765, 0.0747, 0.4308, 0.0844],
        [0.2226, 0.1614, 0.1704, 0.0701, 0.1791, 0.0286, 0.1678]],
       grad_fn=<SoftmaxBackward0>)


<h1> Getting the context vector for each word </h1>

In [51]:
# 7*7 * 7*4 = 7*4
sa = attention @ values
print(f'Self attention layer output is:\n {sa}')


Self attention layer output is:
 tensor([[ 0.2126,  0.1292, -0.4775,  0.1211],
        [ 0.3002,  0.3164, -0.3139,  0.2374],
        [ 0.1347, -0.0497, -0.6528,  0.0128],
        [ 0.6909,  1.0727,  0.2377,  0.7100],
        [ 0.5915,  0.8588,  0.0283,  0.5809],
        [ 0.7568,  1.1891,  0.3040,  0.7829],
        [ 0.1468, -0.0318, -0.6614,  0.0272]], grad_fn=<MmBackward0>)


<h1>Modular Code</h1>


In [52]:
class SelfAttention(nn.Module):
  def __init__(self, D, D_q, D_k, D_v):
    super(SelfAttention, self).__init__()
    self.D = D
    self.D_q = D_q
    self.D_k = D_k
    self.D_v = D_v
    self.W_query = nn.Parameter(torch.rand(D, D_q))
    self.W_keys = nn.Parameter(torch.rand(D, D_k))
    self. W_values = nn.Parameter(torch.rand(D, D_v))

  #forward pass
  def forward(self, X):
    queries = X @ W_query
    keys = X @ W_keys
    values = X @ W_values
    Q_dot_K = queries @ keys.T
    norm_Q_dot_K = Q_dot_K / math.sqrt(D_k)
    attention = F.softmax(norm_Q_dot_K, dim = -1)
    sa = attention @ values
    return sa


In [53]:
sa = SelfAttention(D=4, D_q=2, D_k=2, D_v=4)
cv = sa(sentence_embeddings)
print(cv.shape)
print(cv)

torch.Size([7, 4])
tensor([[ 0.2126,  0.1292, -0.4775,  0.1211],
        [ 0.3002,  0.3164, -0.3139,  0.2374],
        [ 0.1347, -0.0497, -0.6528,  0.0128],
        [ 0.6909,  1.0727,  0.2377,  0.7100],
        [ 0.5915,  0.8588,  0.0283,  0.5809],
        [ 0.7568,  1.1891,  0.3040,  0.7829],
        [ 0.1468, -0.0318, -0.6614,  0.0272]], grad_fn=<MmBackward0>)
